# Graph Embeddings & Machine Learning for Banking

**Predictive Analytics Using Node2Vec Graph Embeddings**

This notebook demonstrates how graph embeddings can be used for:
1. **Fraud Risk Prediction** - Predict risk based on network position
2. **Entity Similarity Search** - Find similar customers/entities
3. **Anomaly Detection** - Identify suspicious subgraphs
4. **Customer Segmentation** - Cluster entities by behavior

## Business Context

Traditional fraud detection focuses on individual transactions or accounts. Graph ML enables:
- **Network-based risk scoring**: An account's risk depends on who it transacts with
- **Structural anomaly detection**: Patterns invisible in transaction logs
- **Predictive clustering**: Identify similar entities before they commit fraud

---

**Author**: David LECONTE - IBM Worldwide | Data & AI | Tiger Team  
**Date**: 2026-03-23  
**Phase**: Use Case #15 - Graph Embeddings/ML

## 1. Setup and Data Generation

In [ ]:
# Standard imports
import sys
import json
from pathlib import Path
from datetime import datetime

# Add project root to path
project_root = Path().absolute().parent.parent
sys.path.insert(0, str(project_root))

# Fix asyncio event loop conflict in Jupyter
import nest_asyncio
nest_asyncio.apply()

# Graph ML imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Project imports
from src.python.client.janusgraph_client import JanusGraphClient
from banking.analytics.graph_ml import (
    GraphMLEngine,
    Node2Vec,
    EmbeddingMethod,
    RiskPrediction,
    create_graph_ml_report
)
from banking.data_generators.scripts.generate_graph_ml_data import (
    generate_graph_ml_data_deterministic,
    load_into_janusgraph
)

# Configure display
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# Memory limits for deterministic ML (configurable via environment)
import os
ML_MAX_VERTICES = int(os.getenv('ML_MAX_VERTICES', '500'))  # Default 500 for stability
ML_EMBEDDING_DIM = int(os.getenv('ML_EMBEDDING_DIM', '64'))

print("✅ Imports loaded successfully")
print(f"Project root: {project_root}")
print(f"ML Limits: max_vertices={ML_MAX_VERTICES}, embedding_dim={ML_EMBEDDING_DIM}")


### 1.1 Generate Deterministic Graph ML Data

In [ ]:
# Generate graph data with known clusters and anomalies
print("Generating deterministic Graph ML data...")

graph_data = generate_graph_ml_data_deterministic(seed=42)

print(f"\n📊 Data Statistics:")
print(f"   Vertices: {graph_data['checksums']['total_vertices']}")
print(f"   Edges: {graph_data['checksums']['total_edges']}")
print(f"   Persons: {graph_data['checksums']['total_persons']}")
print(f"   Companies: {graph_data['checksums']['total_companies']}")
print(f"   Accounts: {graph_data['checksums']['total_accounts']}")

print(f"\n🎯 Ground Truth:")
print(f"   Clusters: {graph_data['statistics']['total_clusters']}")
print(f"   Anomalies: {graph_data['statistics']['total_anomalies']}")
print(f"   Risk Distribution: {graph_data['statistics']['risk_distribution']}")

### 1.2 Load Data into JanusGraph

In [ ]:
# Connect to JanusGraph using auto-detected config
from notebook_config import JANUSGRAPH_CONFIG
from urllib.parse import urlparse

# Parse the URL to get host/port
parsed = urlparse(JANUSGRAPH_CONFIG['url'])
host = parsed.hostname
port = parsed.port

client = JanusGraphClient(
    host=host,
    port=port,
    use_ssl=JANUSGRAPH_CONFIG['use_ssl'],
    verify_certs=False
)
client.connect()

# Verify connection
vertex_count = client.execute("g.V().count()")[0]
edge_count = client.execute("g.E().count()")[0]

print(f"Connected to JanusGraph at {host}:{port}")
print(f"Total vertices: {vertex_count}")
print(f"Total edges: {edge_count}")


## 2. Graph Embeddings with Node2Vec

### 2.1 Understanding Node2Vec

Node2Vec generates node embeddings using biased random walks:

- **p (return parameter)**: Controls likelihood of backtracking
  - p < 1: More local exploration (BFS-like)
  - p > 1: More exploration (DFS-like)

- **q (in-out parameter)**: Controls search depth
  - q < 1: Explores far from source (global structure)
  - q > 1: Stays close to source (local structure)

**For fraud detection, we want balanced exploration** to capture both:
- Local patterns (transaction networks)
- Global structure (community membership)

In [ ]:
# Initialize Graph ML Engine
print("Initializing Graph ML Engine...")

engine = GraphMLEngine(
    client=client,
    embedding_dim=64,  # 64-dimensional embeddings
    seed=42  # Reproducibility
)

print("✅ Engine initialized")

In [ ]:
# Generate embeddings using Node2Vec
print("Generating graph embeddings with Node2Vec...")
print("This may take a moment...")

# Memory check for deterministic execution
vertex_count = client.execute("g.V().count()")[0]
if vertex_count > ML_MAX_VERTICES:
    print(f"\u26a0\ufe0f Limiting to {ML_MAX_VERTICES} vertices (graph has {vertex_count})")
    max_vertices = ML_MAX_VERTICES
else:
    max_vertices = None  # No limit needed

results = engine.embed_graph(
    method=EmbeddingMethod.NODE2VEC,
    walk_length=40,  # Walks of length 40
    num_walks=5,     # 5 walks per node
    p=1.0,           # Balanced return
    q=1.0,           # Balanced exploration
    max_vertices=max_vertices  # Memory limit for determinism
)

print(f"\n✅ Embeddings generated!")
print(f"   Nodes embedded: {results.total_nodes}")
print(f"   Embedding dimension: {results.embedding_dim}")
print(f"   Clusters discovered: {results.total_clusters}")
print(f"   Anomalies detected: {results.anomaly_count}")

## 3. Risk Prediction Analysis

In [ ]:
# Analyze risk predictions
risk_summary = engine.get_risk_summary()

print("📊 Risk Prediction Summary:")
print("-" * 40)
for level, count in risk_summary.items():
    pct = count / results.total_nodes * 100
    bar = "█" * int(pct / 2)
    print(f"   {level.upper():10s}: {count:4d} ({pct:5.1f}%) {bar}")

# Compare with ground truth
print("\n🎯 Ground Truth Comparison:")
print("-" * 40)
gt_dist = graph_data['statistics']['risk_distribution']
print(f"   Ground Truth: {gt_dist}")

In [ ]:
# Examine high-risk entities
print("⚠️  High-Risk Entities Detected:")
print("-" * 70)

high_risk_nodes = [
    (nid, node) 
    for nid, node in results.node_embeddings.items()
    if results.risk_predictions.get(nid) in [RiskPrediction.HIGH, RiskPrediction.CRITICAL]
]

# Sort by risk score
high_risk_nodes.sort(key=lambda x: x[1].risk_score, reverse=True)

for nid, node in high_risk_nodes[:10]:
    risk_level = results.risk_predictions.get(nid, RiskPrediction.LOW).value
    print(f"   {nid[:16]}... | Label: {node.node_label:8s} | Risk: {node.risk_score:.3f} | "
          f"Degree: {node.degree:2d} | Cluster: {node.cluster_id} | {risk_level.upper()}")

## 4. Entity Similarity Search

In [ ]:
# Find similar entities for a sample node
# Pick the first person node
sample_node_id = None
for nid, node in results.node_embeddings.items():
    if node.node_label == "Person":
        sample_node_id = nid
        break

if sample_node_id:
    print(f"🔍 Finding entities similar to: {sample_node_id}")
    print("-" * 70)
    
    similar = engine.find_similar_entities(sample_node_id, k=10)
    
    print(f"\nTop 10 Similar Entities:")
    for other_id, score in similar:
        other_node = results.node_embeddings[other_id]
        print(f"   {other_id[:16]}... | Label: {other_node.node_label:8s} | "
              f"Similarity: {score:.4f} | Risk: {other_node.risk_score:.3f}")

## 5. Cluster Analysis

In [ ]:
# Analyze discovered clusters
print("📊 Cluster Analysis:")
print("-" * 70)

for cluster_id, members in sorted(results.clusters.items()):
    avg_risk = np.mean([results.node_embeddings[nid].risk_score for nid in members])
    avg_degree = np.mean([results.node_embeddings[nid].degree for nid in members])
    
    # Count node types
    types = {}
    for nid in members:
        label = results.node_embeddings[nid].node_label
        types[label] = types.get(label, 0) + 1
    
    type_str = ", ".join([f"{k}: {v}" for k, v in types.items()])
    
    print(f"\n   Cluster {cluster_id}:")
    print(f"      Members: {len(members)}")
    print(f"      Types: {type_str}")
    print(f"      Avg Risk Score: {avg_risk:.3f}")
    print(f"      Avg Degree: {avg_degree:.1f}")

## 6. Anomaly Detection

In [ ]:
# Analyze detected anomalies
print(f"🚨 Anomalies Detected: {len(results.anomalies)}")
print("-" * 70)

if results.anomalies:
    print("\nAnomalous Nodes:")
    for i, anom_id in enumerate(results.anomalies[:10], 1):
        node = results.node_embeddings[anom_id]
        print(f"   {i}. {anom_id[:16]}... | Label: {node.node_label:8s} | "
              f"Degree: {node.degree:2d} | Risk: {node.risk_score:.3f}")
    
    # Compare with ground truth
    gt_anomalies = set(graph_data['ground_truth']['anomalies'])
    detected = set(results.anomalies)
    
    if gt_anomalies:
        overlap = gt_anomalies & detected
        precision = len(overlap) / len(detected) if detected else 0
        recall = len(overlap) / len(gt_anomalies) if gt_anomalies else 0
        
        print(f"\n📈 Detection Performance:")
        print(f"   Ground Truth Anomalies: {len(gt_anomalies)}")
        print(f"   Detected Anomalies: {len(detected)}")
        print(f"   Overlap: {len(overlap)}")
        print(f"   Precision: {precision:.1%}")
        print(f"   Recall: {recall:.1%}")
else:
    print("   No anomalies detected (all nodes clustered normally)")

## 7. Visualization

In [ ]:
# Export for visualization
print("Exporting embeddings for 2D visualization...")

viz_data = engine.export_for_visualization(use_tsne=True, perplexity=15)

print(f"✅ Exported {len(viz_data['nodes'])} nodes and {len(viz_data['edges'])} edges")
print(f"\n📊 Visualization Metadata:")
for k, v in viz_data['metadata'].items():
    print(f"   {k}: {v}")

In [ ]:
# Create 2D scatter plot of embeddings
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: By cluster
ax1 = axes[0]
nodes_df = pd.DataFrame(viz_data['nodes'])

colors = plt.cm.tab20(np.linspace(0, 1, max(nodes_df['cluster']) + 1))
for cluster_id in nodes_df['cluster'].unique():
    cluster_nodes = nodes_df[nodes_df['cluster'] == cluster_id]
    ax1.scatter(
        cluster_nodes['x'], 
        cluster_nodes['y'],
        c=[colors[cluster_id]],
        label=f'Cluster {cluster_id}',
        alpha=0.7,
        s=50
    )

ax1.set_title('Graph Embeddings by Cluster (t-SNE)', fontsize=12)
ax1.set_xlabel('Dimension 1')
ax1.set_ylabel('Dimension 2')
ax1.legend(loc='best', fontsize=8)
ax1.grid(True, alpha=0.3)

# Plot 2: By risk level
ax2 = axes[1]
risk_colors = {
    'low': 'green',
    'medium': 'orange', 
    'high': 'red',
    'critical': 'darkred'
}

for risk_level in ['low', 'medium', 'high', 'critical']:
    risk_nodes = nodes_df[nodes_df['risk_level'] == risk_level]
    ax2.scatter(
        risk_nodes['x'],
        risk_nodes['y'],
        c=risk_colors[risk_level],
        label=risk_level.upper(),
        alpha=0.7,
        s=50
    )

ax2.set_title('Graph Embeddings by Risk Level (t-SNE)', fontsize=12)
ax2.set_xlabel('Dimension 1')
ax2.set_ylabel('Dimension 2')
ax2.legend(loc='best', fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('graph_embeddings_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Visualization saved to graph_embeddings_visualization.png")

## 8. Comprehensive Report

In [ ]:
# Generate comprehensive report
report = create_graph_ml_report(results, output_format="detailed")

print(report)

## 9. Business Applications

### Use Cases for Graph Embeddings in Banking

| Use Case | Description | Business Value |
|----------|-------------|----------------|
| **Fraud Risk Scoring** | Predict risk from network position | Early intervention, reduced losses |
| **KYC Enhancement** | Cluster similar customers | Faster onboarding, better segmentation |
| **Anomaly Detection** | Identify structural outliers | Detect fraud rings, money mules |
| **Customer Similarity** | Find similar entities | Cross-sell, fraud pattern matching |
| **UBO Discovery** | Traverse ownership chains | Regulatory compliance, AML |
| **Transaction Monitoring** | Network-based alerts | Reduce false positives |

### Integration Points

```python
# Real-time fraud scoring API
def score_account_risk(account_id: str) -> float:
    embedding = engine.get_embedding(account_id)
    return engine.predict_risk(embedding)

# Customer similarity for KYC
def find_similar_customers(customer_id: str) -> List[str]:
    return engine.find_similar_entities(customer_id, k=10)

# Anomaly alerting
def check_for_anomalies(new_entity_id: str) -> bool:
    return new_entity_id in engine.get_anomalies()
```

## 10. Summary

In [ ]:
# Summary statistics
print("=" * 70)
print("GRAPH EMBEDDINGS & ML DEMO SUMMARY")
print("=" * 70)

print(f"\n📊 Graph Statistics:")
print(f"   Total nodes embedded: {results.total_nodes}")
print(f"   Embedding dimension: {results.embedding_dim}")
print(f"   Method: {results.method.value}")

print(f"\n🎯 Analysis Results:")
print(f"   Clusters discovered: {results.total_clusters}")
print(f"   Anomalies detected: {results.anomaly_count}")
print(f"   High/Critical risk: {risk_summary['high'] + risk_summary['critical']}")

print(f"\n⚡ Key Findings:")
print(f"   - Graph structure reveals hidden risk patterns")
print(f"   - Embeddings enable similarity-based fraud detection")
print(f"   - Clustering identifies natural customer segments")
print(f"   - Anomaly detection spots suspicious network positions")

print(f"\n✅ Demo complete!")

# Cleanup
client.close()
print("\n🔌 Connection closed.")